In [15]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Modelo principal proposto
### Desfecho
"MG"
"LEVE"

### Variáveis independentes
"ACESSO_LOCAL",    
"TEMPO",  
"INCID_MEDIA_GERAL",  
"PROP_POP_0A10",  
"PROP_POP_60MAIS",  
"LOG_POP",  
"PROP_CLASS_IGN"


### O que queremos descobrir?
* Municípios com maior tempo/distância até o PESA apresentam maior chance de ocorrência de casos moderados/graves?

Controlando por:

- incidência;
- estrutura etária;
- porte populacional;
- qualidade do preenchimento.

In [16]:
df = pd.read_csv(filepath_or_buffer='Dados-modelo/df_preliminar.csv', sep=';')
df = df[df['MG'] + df['LEVE'] > 0].copy()

In [17]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'REGIAO',
       'PESA', 'MUNI_REFERENCIADO', 'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI',
       'LAT_PESA', 'LON_PESA', 'DISTANCIA', 'TEMPO', 'IBGE', 'MUNI_NOME_x',
       'POP10', 'POP12', 'POP11A59', 'POP60', 'POP_GERAL', 'MACRO_CODIGO',
       'MACRO_NOME', 'DRS_CODIGO', 'DRS_NOME', 'CIR_CODIGO', 'CIR_NOME',
       'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_12A', 'TOTAL_11A59', 'TOTAL_60',
       'LEVE', 'MG', 'LEVE_10', 'MG_10', 'LEVE_12', 'MG_12', 'LEVE_11A59',
       'MG_11A59', 'LEVE_60', 'MG_60', 'TOTAL_AMPOLAS', 'CAT_TEMPO',
       'PROP_POP_0A10', 'PROP_POP_60MAIS', 'LOG_POP', 'PROP_MG',
       'INCID_MEDIA_GERAL', 'INCID_0A10', 'INCID_60MAIS', 'PROP_CASOS_0A10',
       'PROP_CASOS_60MAIS', 'TAXA_MG_100MIL', 'CLASS_IG', 'SORO_IG', 'EVOL_IG',
       'PROP_CLASS_IGN', 'PROP_SORO_IGN', 'PROP_EVOL_IGN', 'BAIXO_N'],
      dtype='str')

In [18]:
# Modelo 1 (sem distancia)
modelo1 = smf.glm(
    formula="""
        MG + LEVE ~
        TEMPO +
        INCID_MEDIA_GERAL +
        PROP_POP_0A10 +
        PROP_POP_60MAIS +
        LOG_POP +
        PROP_CLASS_IGN
    """,
    data=df,
    family=sm.families.Binomial()
).fit()

In [19]:
print(modelo1.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         ['MG', 'LEVE']   No. Observations:                  635
Model:                            GLM   Df Residuals:                      628
Model Family:                Binomial   Df Model:                            6
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2286.8
Date:                sáb, 11 jul 2026   Deviance:                       3067.5
Time:                        23:36:34   Pearson chi2:                 4.05e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.4919
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -2.7561      0.48

In [ ]:
# Transformar em Oddsratio
np.exp(modelo1.params)

# Interpretação: Cada minuto adicional até o PESA aumenta em 1,3% a chance de um caso ser moderado/grave.

Intercept              0.063540
TEMPO                  0.985997
INCID_MEDIA_GERAL      0.998507
PROP_POP_0A10        460.512479
PROP_POP_60MAIS        6.059499
LOG_POP                0.782449
PROP_CLASS_IGN         3.701670
dtype: float64

In [22]:
# Verificar multicolinearidade
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[
    [
        'TEMPO',
        'INCID_MEDIA_GERAL',
        'PROP_POP_0A10',
        'PROP_POP_60MAIS',
        'LOG_POP',
        'PROP_CLASS_IGN'
    ]
]

X = sm.add_constant(X)

vif = pd.DataFrame()

vif["Variavel"] = X.columns

vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif)

            Variavel         VIF
0              const  456.961366
1              TEMPO    1.422762
2  INCID_MEDIA_GERAL    1.248993
3      PROP_POP_0A10    2.045836
4    PROP_POP_60MAIS    2.332215
5            LOG_POP    1.796907
6     PROP_CLASS_IGN    1.050393


In [23]:
# Municípios com acesso local apresentam menos MG do que municípios a mais de 60 minutos?
modelo_cat = smf.glm(

    formula="""
        MG + LEVE ~
        C(CAT_TEMPO)
    """,

    data=df,

    family=sm.families.Binomial()

).fit()

print(modelo_cat.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         ['MG', 'LEVE']   No. Observations:                  635
Model:                            GLM   Df Residuals:                      631
Model Family:                Binomial   Df Model:                            3
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2483.1
Date:                sáb, 11 jul 2026   Deviance:                       3460.2
Time:                        23:44:12   Pearson chi2:                 5.39e+03
No. Iterations:                     8   Pseudo R-squ. (CS):            0.05707
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 